# 💾 Checkpointing and Persistence in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Checkpointing Basics** - How LangGraph automatically saves a snapshot of graph state after every node runs, using a checkpointer
2. **In-Memory vs. Durable Storage** - The difference between `MemorySaver` (development, lost on restart) and `SqliteSaver` (durable, survives process restarts)
3. **State Inspection** - How to read the current state of a thread and walk its full checkpoint history
4. **Branching Conversations** - How to fork a conversation into independent threads starting from the same checkpoint
5. **Checkpoint Internals** - What fields (`values`, `next`, `config`, `metadata`, `parent_config`, `created_at`) make up a checkpoint, and how to rewind to a past one

## Prerequisites
- Familiarity with LangGraph's `StateGraph`, nodes, and edges (see `01_Foundations`)
- An `OPENAI_API_KEY` set in a `.env` file at the project root
- Basic understanding of Python's `TypedDict` and `Annotated` types

---
## 📦 Part 0: Environment Setup

Before building any graphs, we load environment variables and initialize the LLM that every demo in this notebook shares. Checkpointing itself is a graph-compilation concern (the `checkpointer=` argument to `.compile()`), so the setup here is intentionally minimal.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LLM Initialization
# ============================================================================
import operator
import tempfile

from dotenv import load_dotenv
from typing_extensions import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

# Load API keys from .env at the project root
load_dotenv()

# Initialize the LLM used by every demo in this notebook
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

print(f"🤖 LLM initialized: {llm.model_name}")
print("✅ Environment ready!")

🤖 LLM initialized: gpt-4o-mini
✅ Environment ready!


---
## 🗂️ Part 1: Defining Shared State

Every graph in this notebook shares the same minimal state schema: a running list of chat messages. The `operator.add` reducer tells LangGraph to **append** new messages returned by a node to the existing list, rather than overwriting it - this is what lets a checkpointed thread accumulate conversation history across multiple `.invoke()` calls.

### Key Concepts:
- **Checkpointer**: An object passed to `graph.compile(checkpointer=...)` that saves a snapshot of state after every node runs
- **Thread**: A conversation identified by a `thread_id` in the config - each thread has its own independent checkpoint history
- **Reducer**: A function (here `operator.add`) that merges a node's returned update into the existing state field

In [2]:
# ============================================================================
# CHAT STATE: Shared Schema for All Checkpointing Demos
# ============================================================================
class ChatState(TypedDict):
    # operator.add appends new messages instead of replacing the list
    messages: Annotated[list[BaseMessage], operator.add]

---
## 🧠 Part 2: In-Memory Checkpointing

`MemorySaver` keeps checkpoints in process memory - fast and dependency-free, making it ideal for local development and testing. Its one limitation: state is lost the moment the Python process exits. This demo shows a single graph resuming a multi-turn conversation purely because both calls share the same `thread_id`.

In [3]:
# ============================================================================
# IN-MEMORY CHECKPOINTING: MemorySaver for Development
# ============================================================================
def demo_memory_saver():
    """In-memory checkpointing for development."""

    def chat(state: ChatState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(ChatState)
    graph.add_node("chat", chat)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)

    saver = MemorySaver()
    app = graph.compile(checkpointer=saver)

    # Configuration with thread_id
    config = {"configurable": {"thread_id": "user-123"}}

    print("Memory Saver Demo (Multi-turn conversation):\n")

    # Turn 1
    result = app.invoke(
        {"messages": [HumanMessage(content="My name is Paulo")]}, config
    )
    print(f"Turn 1 - AI: {result['messages'][-1].content}")

    # Turn 2 - Conversation continues
    result = app.invoke({"messages": [HumanMessage(content="What's my name?")]}, config)
    print(f"Turn 2 - AI: {result['messages'][-1].content}")

    # Check full history
    state = app.get_state(config)
    print(f"\nTotal messages in state: {len(state.values['messages'])}")

---
## 🗄️ Part 3: SQLite Persistence

`SqliteSaver` writes checkpoints to a SQLite database file instead of memory, so state **survives process restarts**. This demo simulates that: it opens a session, stores a fact, closes the connection, then reopens a brand-new `SqliteSaver` session against the same database file and shows the graph can still recall the fact.

> **Note**: For production multi-user systems, LangGraph also ships a `PostgresSaver` with the same interface - swap it in when SQLite's single-writer model becomes a bottleneck.

In [4]:
# ============================================================================
# SQLITE PERSISTENCE: Durable Storage Across Sessions
# ============================================================================
def demo_sqlite_persistence():
    """SQLite persistence for durable storage."""

    def chat(state: ChatState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(ChatState)
    graph.add_node("chat", chat)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)

    # Create temp database
    with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
        db_path = f.name

    print("\nSQLite Persistence Demo:")
    print(f"Database: {db_path}\n")

    # First session
    with SqliteSaver.from_conn_string(db_path) as saver:
        app = graph.compile(checkpointer=saver)
        config = {"configurable": {"thread_id": "persistent-user"}}
        result = app.invoke(
            {
                "messages": [
                    HumanMessage(content="Remember: The secret code is ALPHA-7")
                ]
            },
            config,
        )
        print("Session 1 - Stored secret code")
        # PostgresSaver with a real database!

    # Simulate app restart - new session
    with SqliteSaver.from_conn_string(db_path) as saver:
        app = graph.compile(checkpointer=saver)
        config = {"configurable": {"thread_id": "persistent-user"}}
        result = app.invoke(
            {"messages": [HumanMessage(content="What was the secret code?")]}, config
        )
        print(f"Session 2 - AI: {result['messages'][-1].content}")

---
## 🔍 Part 4: Inspecting Checkpoint State

Beyond just resuming a conversation, LangGraph lets you actively inspect a thread's state at any point. `app.get_state(config)` returns the **latest** snapshot; `app.get_state_history(config)` walks **every** checkpoint ever saved for that thread, newest first.

In [5]:
# ============================================================================
# STATE INSPECTION: Reading Current State and Checkpoint History
# ============================================================================
def demo_state_inspection():
    """Inspect and manipulate checkpoint state."""

    def chat(state: ChatState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(ChatState)
    graph.add_node("chat", chat)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)

    memory = MemorySaver()
    app = graph.compile(checkpointer=memory)
    config = {"configurable": {"thread_id": "inspect-demo"}}

    print("\nState Inspection Demo:\n")

    # Build up some state
    app.invoke({"messages": [HumanMessage(content="Hello!")]}, config)
    app.invoke({"messages": [HumanMessage(content="How are you?")]}, config)

    # Get current state
    state = app.get_state(config)
    print("Current state:")
    print(f"  Next node: {state.next}")
    print(f"  Message count: {len(state.values['messages'])}")

    # Get state history
    print("\nState history:")
    for i, snapshot in enumerate(app.get_state_history(config)):
        print(f"  Checkpoint {i}: {len(snapshot.values['messages'])} messages")
        if i >= 3:
            print("  ...")
            break

---
## 🌿 Part 5: Branching Conversations from a Checkpoint

Because a checkpoint is just a snapshot of state, you can copy it into a brand-new `thread_id` with `app.update_state()` and continue from there independently. This demo takes one shared conversation starting point and forks it into two unrelated branches - a beach-vacation thread and a mountain-hiking thread - each with its own checkpoint history from that point forward.

### Key Insight:
> Branching doesn't mutate the original thread. `main_state.values` is copied into each new thread's initial checkpoint, so `main`, `branch-beach`, and `branch-mountain` all evolve independently afterward.

In [6]:
# ============================================================================
# BRANCHING CONVERSATIONS: Forking a Checkpoint into New Threads
# ============================================================================
def demo_branching_conversations():
    """Branch conversations from checkpoints."""

    def chat(state: ChatState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(ChatState)
    graph.add_node("chat", chat)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)

    memory = MemorySaver()
    app = graph.compile(checkpointer=memory)

    print("\nBranching Conversations Demo:\n")

    # Main conversation
    main_config = {"configurable": {"thread_id": "main"}}
    app.invoke(
        {"messages": [HumanMessage(content="What's the weather like?")]}, main_config
    )

    # Get checkpoint to branch from
    main_state = app.get_state(main_config)

    # Branch A - Beach vacation
    branch_a_config = {"configurable": {"thread_id": "branch-beach"}}
    # Copy state to new thread
    app.update_state(branch_a_config, main_state.values)
    result_a = app.invoke(
        {"messages": [HumanMessage(content="What about a beach vacation?")]},
        branch_a_config,
    )
    print(f"Branch A (Beach): {result_a['messages'][-1].content[:100]}...")

    # Branch B - Mountain adventure
    branch_b_config = {"configurable": {"thread_id": "branch-mountain"}}
    app.update_state(branch_b_config, main_state.values)
    result_b = app.invoke(
        {"messages": [HumanMessage(content="What about mountain hiking?")]},
        branch_b_config,
    )
    print(f"Branch B (Mountain): {result_b['messages'][-1].content[:100]}...")

---
## 🧬 Part 6: Checkpoint Internals (Anatomy of a Checkpoint)

So far we have treated checkpoints as an opaque mechanism. This demo builds a 2-node graph (`analyze` -> `summarize`) so multiple checkpoints get created, then opens up every field on a state snapshot: `values`, `next`, `config`, `metadata`, `parent_config`, and `created_at`. It finishes by demonstrating **time travel** - jumping back to the checkpoint saved right after `analyze` ran, before `summarize` executed.

### Key Concepts:
- **`state.values`**: Your actual `TypedDict` data at this checkpoint
- **`state.next`**: The node(s) that will run next (`()` means the graph has finished)
- **`state.config`**: The `thread_id` + `checkpoint_id` that uniquely address this snapshot
- **`state.parent_config`**: A pointer to the previous checkpoint, forming a linked list
- **`state.metadata`**: Provenance - which node wrote this checkpoint and at what step
- **`state.created_at`**: When the checkpoint was saved

> **Note**: Checkpoints are saved before the first node runs, after every node completes, and at interrupt points used for human-in-the-loop workflows.

In [7]:
# ============================================================================
# CHECKPOINT INTERNALS: Anatomy of a Checkpoint and Time Travel
# ============================================================================
def demo_checkpoint_internals():
    """
    Peek inside a checkpoint - see exactly what LangGraph saves.
    Uses a 2-node graph so we generate multiple checkpoints,
    then walks through every field in the checkpoint object.
    """

    # -- Build a 2-node graph so we get several checkpoints --
    class TaskState(TypedDict):
        messages: Annotated[list[BaseMessage], operator.add]
        step: str

    def analyze(state: TaskState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response], "step": "analyzed"}

    def summarize(state: TaskState) -> dict:
        summary_prompt = [
            HumanMessage(
                content=f"Summarize this in one sentence: {state['messages'][-1].content}"
            )
        ]
        response = llm.invoke(summary_prompt)
        return {"messages": [response], "step": "summarized"}

    graph = StateGraph(TaskState)
    graph.add_node("analyze", analyze)
    graph.add_node("summarize", summarize)
    graph.add_edge(START, "analyze")
    graph.add_edge("analyze", "summarize")
    graph.add_edge("summarize", END)

    memory = MemorySaver()
    app = graph.compile(checkpointer=memory)
    config = {"configurable": {"thread_id": "internals-demo"}}

    print("\nCheckpoint Internals Demo")
    print("=" * 55)
    print("Graph: START -> [analyze] -> [summarize] -> END")
    print("=" * 55)

    # -- Run the graph --
    app.invoke(
        {"messages": [HumanMessage(content="Explain why the sky is blue")], "step": ""},
        config,
    )

    # ========================================================
    # PART 1: What's in the CURRENT state snapshot?
    # ========================================================
    print("\n--- PART 1: Current State Snapshot (app.get_state) ---\n")
    state = app.get_state(config)

    # state.values - your actual TypedDict data
    print("1) state.values (your state data):")
    print(f"   step: '{state.values['step']}'")
    print(f"   messages: {len(state.values['messages'])} total")
    for i, msg in enumerate(state.values["messages"]):
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"     [{i}] {role}: {msg.content[:80]}...")

    # state.next - which node runs next (empty = graph finished)
    print("\n2) state.next (pending node):")
    print(f"   {state.next if state.next else '() -- graph finished, no pending nodes'}")

    # state.config - the config that produced this snapshot
    print("\n3) state.config (thread + checkpoint IDs):")
    print(f"   thread_id:     {state.config['configurable']['thread_id']}")
    print(f"   checkpoint_id: {state.config['configurable']['checkpoint_id']}")

    # state.metadata - who created this checkpoint
    print("\n4) state.metadata (provenance info):")
    print(f"   source:  {state.metadata.get('source', 'N/A')}")
    print(f"   step:    {state.metadata.get('step', 'N/A')}")
    print(f"   writes:  {state.metadata.get('writes', 'N/A')}")

    # state.parent_config - pointer to the PREVIOUS checkpoint
    print("\n5) state.parent_config (previous checkpoint):")
    if state.parent_config:
        print(
            f"   parent checkpoint_id: {state.parent_config['configurable']['checkpoint_id']}"
        )
    else:
        print("   None -- this is the very first checkpoint")

    # state.created_at - timestamp
    print("\n6) state.created_at (when saved):")
    print(f"   {state.created_at}")

    # ========================================================
    # PART 2: Walk through ALL checkpoints (time travel)
    # ========================================================
    print("\n--- PART 2: Full Checkpoint History (app.get_state_history) ---\n")
    print("LangGraph saves a checkpoint at EACH step. Let's see them all:\n")

    for i, snapshot in enumerate(app.get_state_history(config)):
        step_num = snapshot.metadata.get("step", "?")
        source = snapshot.metadata.get("source", "?")
        writes = snapshot.metadata.get("writes", {})
        msg_count = len(snapshot.values.get("messages", []))
        checkpoint_id = snapshot.config["configurable"]["checkpoint_id"]
        current_step = snapshot.values.get("step", "")

        # Which node just wrote to this checkpoint?
        node_name = list(writes.keys())[0] if writes else "--"

        print(f"  Checkpoint {i}:")
        print(f"    id:         {checkpoint_id[:30]}...")
        print(f"    source:     {source}")
        print(f"    step:       {step_num}")
        print(f"    written by: {node_name}")
        print(f"    state.step: '{current_step}'")
        print(f"    messages:   {msg_count}")
        print(f"    next:       {snapshot.next if snapshot.next else '() -- finished'}")
        print(f"    created_at: {snapshot.created_at}")
        print()

    # ========================================================
    # PART 3: Jump to a specific checkpoint (rewind)
    # ========================================================
    print("--- PART 3: Rewind -- Jump to a Previous Checkpoint ---\n")

    # Find the checkpoint right after the "analyze" node ran
    target_snapshot = None
    for snapshot in app.get_state_history(config):
        writes = snapshot.metadata.get("writes", {})
        if "analyze" in writes:
            target_snapshot = snapshot
            break

    if target_snapshot:
        target_id = target_snapshot.config["configurable"]["checkpoint_id"]
        print(f"  Found checkpoint after 'analyze' node: {target_id[:30]}...")
        print(f"  Messages at that point: {len(target_snapshot.values['messages'])}")
        print(f"  state.step at that point: '{target_snapshot.values.get('step', '')}'")

        # You can resume from this exact checkpoint
        rewind_config = {
            "configurable": {"thread_id": "internals-demo", "checkpoint_id": target_id}
        }
        rewound_state = app.get_state(rewind_config)
        print(f"\n  Loaded checkpoint -- next node would be: {rewound_state.next}")
        print("  We're back to BEFORE 'summarize' ran!")
        print(
            "  Calling invoke(None) from here would re-run 'summarize' with fresh output."
        )
    else:
        print("  Could not find target checkpoint.")

    # ========================================================
    # SUMMARY: Anatomy of a checkpoint
    # ========================================================
    print("\n" + "=" * 55)
    print("  CHECKPOINT ANATOMY -- What Gets Saved")
    print("=" * 55)
    print(
        """
    state.values        -> Your TypedDict data (messages, step, etc.)
    state.next          -> Tuple of nodes that run next (() if done)
    state.config        -> thread_id + checkpoint_id (unique address)
    state.parent_config -> Previous checkpoint's address (linked list)
    state.metadata      -> source, step number, which node wrote
    state.created_at    -> Timestamp of when this checkpoint was saved

    Checkpoints are saved:
      1. BEFORE the first node runs (initial input state)
      2. AFTER each node completes (with updated state)
      3. At interrupt points (frozen state for human-in-the-loop)

    Think of it as a linked list of snapshots:
      [initial] --> [after analyze] --> [after summarize]
         ^               ^                    ^
       parent          parent              current (latest)
    """
    )

---
## ▶️ Running the Demos

Each demo above is a standalone function - uncomment the one you want to run. Only `demo_checkpoint_internals()` is uncommented by default since it is the most comprehensive walkthrough, covering state inspection and rewinding in one place.

In [8]:
# ============================================================================
# RUN: Execute a Checkpointing Demo
# ============================================================================
if __name__ == "__main__":
    # demo_memory_saver()
    # demo_sqlite_persistence()
    # demo_state_inspection()
    # demo_branching_conversations()
    demo_checkpoint_internals()


Checkpoint Internals Demo
Graph: START -> [analyze] -> [summarize] -> END

--- PART 1: Current State Snapshot (app.get_state) ---

1) state.values (your state data):
   step: 'summarized'
   messages: 3 total
     [0] Human: Explain why the sky is blue...
     [1] AI: The sky appears blue primarily due to a phenomenon called Rayleigh scattering. T...
     [2] AI: The sky appears blue due to Rayleigh scattering, where shorter blue wavelengths ...

2) state.next (pending node):
   () -- graph finished, no pending nodes

3) state.config (thread + checkpoint IDs):
   thread_id:     internals-demo
   checkpoint_id: 1f1a9742-5381-6e72-8002-a338b3e422f8

4) state.metadata (provenance info):
   source:  loop
   step:    2
   writes:  N/A

5) state.parent_config (previous checkpoint):
   parent checkpoint_id: 1f1a9742-48de-63be-8001-4b80ca1b62ac

6) state.created_at (when saved):
   2026-09-05T21:52:53.093131+00:00

--- PART 2: Full Checkpoint History (app.get_state_history) ---

LangGraph sav

---
## 📝 Summary

In this notebook, we learned:

### 1. Checkpointing Fundamentals
- **`graph.compile(checkpointer=...)`** turns any graph into a stateful, resumable one
- **`MemorySaver`**: fast, dependency-free, but lost when the process exits - use it for development
- **`SqliteSaver`**: durable storage in a SQLite file, survives process restarts - a `PostgresSaver` exists for production multi-user systems with the same interface
- **`thread_id`** in the config is what identifies a conversation - two calls with the same `thread_id` share history, different `thread_id`s are fully independent

### 2. Inspecting and Manipulating State
- **`app.get_state(config)`** returns the latest checkpoint snapshot for a thread
- **`app.get_state_history(config)`** walks every checkpoint ever saved for that thread, newest first
- **`app.update_state(config, values)`** can seed a brand-new thread with an existing snapshot's values, enabling conversation branching

### 3. Checkpoint Anatomy
- A checkpoint bundles `values` (your state), `next` (pending nodes), `config` (thread + checkpoint IDs), `parent_config` (the previous checkpoint), `metadata` (which node wrote it), and `created_at`
- Checkpoints form a linked list via `parent_config`, which is what makes **time travel** possible - resuming from any past checkpoint by passing its `checkpoint_id` in the config

### Next Steps
- Explore how checkpointing combines with **human-in-the-loop** interrupts to pause a graph mid-run for approval
- Look at `PostgresSaver` for production-grade, multi-user persistent checkpointing